In [1]:
import os
import csv
import time
import threading
import multiprocessing
import queue
import requests
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from alphagenome import colab_utils
from alphagenome.models import dna_client, variant_scorers
from alphagenome.data import transcript as transcript_utils
from alphagenome.data import gene_annotation, genome, transcript, track_data
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9'
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["TF_GPU_ALLOCATOR"]='cuda_malloc_async'

import matplotlib.pyplot as plt
import pandas as pd
import pysam
from pysam import VariantFile
from io import StringIO
from tqdm import tqdm
import os
# from dotenv import load_dotenv

pd.set_option('display.max_columns', None)


load_dotenv(dotenv_path=".env")


True

In [2]:
LMNA_INTERVAL = genome.Interval('chr1', 156_082_572, 156_140_081)
156111327 + 500000

156611327

In [3]:
seq_interval = LMNA_INTERVAL.resize(1_048_576)
seq_interval

Interval(chromosome='chr1', start=155587039, end=156635615, strand='.', name='')

In [11]:
BASE_PATH = '/users/PAS2905/coraalbers/'
AG_DATA_PATH = '/users/PAS2905/coraalbers/ag/ag_data/'

GENE_STRAND = "+"
GENE_NAME   = "LMNA"
CHROM       = "chr1"
ISM_START = 156_082_572    
ISM_END   = 156_082_575
CHUNK_BP        = 500   # bp per score_ism_variants() call
SEQ_LENGTH      = 1_048_576 #model input/context window
CALL_TIMEOUT_S  = 1000
MAX_WORKERS     = 16    # parallelism on AG's end. internal score_ism_variants concurrency per call
TISSUE_ONTOLOGY = "UBERON:0002084"  # LV
SCORES_CSV = str(f"lmna_ism_test{ISM_START}_{ISM_END}_{SEQ_LENGTH}.csv")
# Dynamically picks up every non-empty AG_API_KEY_<N> env var present, in
# numeric order -- add or remove keys in .env and the worker count follows,
API_KEYS = [v for _, v in sorted(
    ((k, os.environ[k].strip()) for k in os.environ
     if k.startswith("AG_API_KEY_") and k.removeprefix("AG_API_KEY_").isdigit()),
    key=lambda kv: int(kv[0].removeprefix("AG_API_KEY_")),
) if v]

api_key = API_KEYS[3]

ccre_bed_path = f'{BASE_PATH}ag/variant-effects/osc/data_sync/predicted_ccre_like_regions_central_LV_128res.bed'

In [7]:
all_scorers = variant_scorers.RECOMMENDED_VARIANT_SCORERS
scorer_titles = ['ATAC', 'DNASE', 'CHIP_TF', 'CHIP_HISTONE', 'CAGE', 'PROCAP', 'RNA_SEQ', 'RNA_SEQ_ACTIVE', 'ATAC_ACTIVE', 'DNASE_ACTIVE', 'CHIP_TF_ACTIVE', 'CHIP_HISTONE_ACTIVE', 'CAGE_ACTIVE', 'PROCAP_ACTIVE', 'SPLICE_SITES', 'SPLICE_SITE_USAGE', 'SPLICE_JUNCTIONS']

# print(*all_scorers, sep='\n')
for key, value in all_scorers.items():
    print(f"{key}: {value}")

ATAC: CenterMaskScorer(requested_output=ATAC, width=501, aggregation_type=DIFF_LOG2_SUM)
CONTACT_MAPS: ContactMapScorer()
DNASE: CenterMaskScorer(requested_output=DNASE, width=501, aggregation_type=DIFF_LOG2_SUM)
CHIP_TF: CenterMaskScorer(requested_output=CHIP_TF, width=501, aggregation_type=DIFF_LOG2_SUM)
CHIP_HISTONE: CenterMaskScorer(requested_output=CHIP_HISTONE, width=2001, aggregation_type=DIFF_LOG2_SUM)
CAGE: CenterMaskScorer(requested_output=CAGE, width=501, aggregation_type=DIFF_LOG2_SUM)
PROCAP: CenterMaskScorer(requested_output=PROCAP, width=501, aggregation_type=DIFF_LOG2_SUM)
RNA_SEQ: GeneMaskLFCScorer(requested_output=RNA_SEQ)
RNA_SEQ_ACTIVE: GeneMaskActiveScorer(requested_output=RNA_SEQ)
SPLICE_SITES: GeneMaskSplicingScorer(requested_output=SPLICE_SITES, width=None)
SPLICE_SITE_USAGE: GeneMaskSplicingScorer(requested_output=SPLICE_SITE_USAGE, width=None)
SPLICE_JUNCTIONS: SpliceJunctionScorer()
POLYADENYLATION: PolyadenylationScorer()
ATAC_ACTIVE: CenterMaskScorer(reques

In [9]:
ccre_df = pd.read_csv(ccre_bed_path, sep='\t')
ccre_df['start'] = ccre_df['start'].astype(int)
ccre_df['end'] = ccre_df['end'].astype(int)
ccre_df.head()

,chrom,start,end,name,score,cCRE_class,d_tss,high_CA,high_H3K4me3,high_H3K27ac,high_CTCF,high_TF,max_CA,max_H3K4me3,max_H3K27ac,max_CTCF,max_TF
0,chr1,156338015,156338911,pred_ccre_central_0_Distal enhancer,19.720718,Distal enhancer,200763,True,True,True,True,True,19.720718,7.782745,3.437913,0.435852,9.092731
1,chr1,156052959,156055391,pred_ccre_central_1_Distal enhancer,18.748081,Distal enhancer,28396,True,True,True,True,True,18.748081,8.692101,7.210982,9.786079,7.372068
2,chr1,156456415,156457823,pred_ccre_central_2_Distal enhancer,18.621325,Distal enhancer,319419,True,True,True,True,True,18.621325,5.600292,10.808559,11.645118,8.048043
3,chr1,156282335,156283103,pred_ccre_central_3_Distal enhancer,17.011522,Distal enhancer,145019,True,True,True,True,True,17.011522,9.146778,5.850923,0.362469,12.718414
4,chr1,156193503,156194271,pred_ccre_central_4_Distal enhancer,16.304649,Distal enhancer,56187,True,True,True,True,True,16.304649,7.464471,3.459850,6.997523,6.941903


In [12]:
scorer = 'RNA_SEQ'

ccre_row = ccre_df.iloc[0]
chunk_start = ccre_row['start']
chunk_end = ccre_row['end']

model = dna_client.create(api_key)
ism_interval = genome.Interval(CHROM, chunk_start, chunk_end)
LMNA_INTERVAL = genome.Interval('chr1', 156_082_572, 156_140_081)
seq_interval = LMNA_INTERVAL.resize(SEQ_LENGTH)



result = model.score_ism_variants(
            interval=seq_interval,
            ism_interval=ism_interval,
            variant_scorers=[all_scorers[scorer]],
            organism=dna_client.Organism.HOMO_SAPIENS,
            progress_bar=False,
            max_workers=MAX_WORKERS,
            merge_stranded_gene_tracks=False,
        )

In [18]:
result[0][0].obs

,gene_id,strand,gene_name,gene_type
0,ENSG00000116580.20,-,GON4L,protein_coding
1,ENSG00000116584.22,-,ARHGEF2,protein_coding
2,ENSG00000116586.12,+,LAMTOR2,protein_coding
3,ENSG00000116604.19,-,MEF2D,protein_coding
4,ENSG00000125459.18,+,MSTO1,protein_coding
5,ENSG00000125462.20,-,MIR9-1HG,lncRNA
6,ENSG00000132676.16,+,DAP3,protein_coding
7,ENSG00000132677.13,+,RHBG,protein_coding
8,ENSG00000132680.11,-,KHDC4,protein_coding
9,ENSG00000132698.15,+,RAB25,protein_coding


In [7]:
def fetch_reference_sequence(chrom_bare: str, start_1based: int, end_1based: int) -> str:
    url = (f"https://rest.ensembl.org/sequence/region/human/"
           f"{chrom_bare}:{start_1based}-{end_1based}")
    r = requests.get(url, headers={"Content-Type": "application/json"}, timeout=30)
    r.raise_for_status()
    return r.json()["seq"].upper()

def build_all_variant_rows(ref_seq: str, start_1based: int, score_cols: list) -> list:
    blank_scores = {c: "" for c in score_cols}
    return [
        {"position": start_1based + i, "ref": ref, "alt": alt, **blank_scores}
        for i, ref in enumerate(ref_seq)
        for alt in ([b for b in "ACGT" if b != ref] if ref in "ACGT" else list("ACGT"))
    ]

# extract_score's shape guards double as emptiness guards: an empty adata.X
# yields an all-False col_mask/row_mask below, which the .any() checks catch.
def extract_score(adata, output_type_name: str) -> float:
    """Reduce one ISM AnnData to a single signed score for this variant.

    `score_ism_variants` returns one AnnData per scorer. Rows (`obs`) are
    genes (or bins, for contact maps); columns (`var`) are tracks. This
    function subsets to the configured gene / tissue / assay, then keeps
    the finite value with the largest absolute magnitude so a strong
    signed effect is not cancelled by averaging.

    Returns NaN if the subset is empty or all-non-finite (no matching
    track, gene missing from the scored interval, etc.).
    """
    # X: (n_genes_or_bins, n_tracks); var: track metadata for columns.
    X, var = adata.X, adata.var

    # --- Column mask: which tracks to keep ---
    # Splice-site class probabilities are not tissue-specific. Keep only
    # donor/acceptor tracks on GENE_STRAND (must match the gene's transcribed
    # strand; LMNA is '-', so GENE_STRAND needs to match that).
    if output_type_name == "SPLICE_SITES":
        col_mask = ((var["name"] == "donor") | (var["name"] == "acceptor")) & \
                   (var["strand"] == GENE_STRAND)
    # Junctions are tissue + assay specific but unstranded in metadata.
    # Restrict to left ventricle (TISSUE_ONTOLOGY) total RNA-seq.
    elif output_type_name == "SPLICE_JUNCTIONS":
        col_mask = (var["ontology_curie"] == TISSUE_ONTOLOGY) & \
                   (var["Assay title"] == "total RNA-seq")
    # Splice-site usage is tissue-, assay-, and strand-specific.
    elif output_type_name == "SPLICE_SITE_USAGE":
        col_mask = (var["ontology_curie"] == TISSUE_ONTOLOGY) & \
                   (var["Assay title"] == "total RNA-seq") & \
                   (var["strand"] == GENE_STRAND)
    # Contact maps have no per-track ontology filter here; use all columns.
    elif output_type_name == "CONTACT_MAPS":
        col_mask = np.ones(X.shape[1], dtype=bool)
    # ATAC/DNASE/CAGE/RNA_SEQ/ChIP/etc.: one (or more) tracks per ontology.
    elif "ontology_curie" in var.columns:
        col_mask = (var["ontology_curie"] == TISSUE_ONTOLOGY)
    else:
        # No ontology column (rare); fall back to every track.
        col_mask = np.ones(X.shape[1], dtype=bool)
    # Boolean Series -> numpy mask so X indexing and .any() work uniformly.
    col_mask = col_mask.values if hasattr(col_mask, "values") else col_mask
    if not col_mask.any():
        return float("nan")

    # --- Row mask: which genes/bins to keep ---
    obs = adata.obs
    if output_type_name == "CONTACT_MAPS":
        # Contact-map rows are spatial bins, not genes.
        row_mask = np.ones(X.shape[0], dtype=bool)
    else:
        # Gene-mask scorers (RNA, splicing) score each overlapping gene;
        # keep only GENE_NAME (LMNA). Center-mask scorers may lack gene_name.
        row_mask = (obs["gene_name"] == GENE_NAME).values if "gene_name" in obs.columns \
            else np.ones(X.shape[0], dtype=bool)
    if not row_mask.any():
        return float("nan")

    # Outer-product slice: selected genes × selected tracks, then flatten.
    sub = np.asarray(X)[np.ix_(row_mask, col_mask)].flatten().astype(float)
    finite = sub[np.isfinite(sub)]
    if finite.size == 0:
        return float("nan")
    # Largest |effect| among remaining values, preserving sign.
    return float(finite[np.argmax(np.abs(finite))])

def run_one_piece(api_key, chunk_start, chunk_end, out_queue):
    from alphagenome.data import genome
    from alphagenome.models import dna_client

    scorers = _build_scorers()
    model = dna_client.create(api_key)
    ism_interval = genome.Interval(CHROM, chunk_start, chunk_end)
    LMNA_INTERVAL = genome.Interval('chr1', 156_082_572, 156_140_081)
    seq_interval = LMNA_INTERVAL.resize(SEQ_LENGTH)

    t0 = time.time()
    try:
        result = model.score_ism_variants(
            interval=seq_interval,
            ism_interval=ism_interval,
            variant_scorers=[s for _, _, s in scorers],
            organism=dna_client.Organism.HOMO_SAPIENS,
            progress_bar=False,
            max_workers=MAX_WORKERS,
            merge_stranded_gene_tracks=False,
        )
        partial = {}
        for variant_scores in result:
            v = variant_scores[0].uns["variant"]
            key = (v.position, v.reference_bases, v.alternate_bases)
            row = {}
            for (col_name, ot_name, _scorer), adata in zip(scorers, variant_scores):
                score = extract_score(adata, ot_name)
                row[col_name] = "NaN" if np.isnan(score) else score
            partial[key] = row
        out_queue.put({"status": "ok", "elapsed": time.time() - t0, "partial": partial})
    except Exception:
        out_queue.put({"status": "error", "elapsed": time.time() - t0, "partial": {}})

def key_dispatcher(worker_id, api_key, my_chunks, rows_by_position,
                    lock, write_csv_fn, counters):
    ctx = multiprocessing.get_context("spawn")

    for chunk_start, chunk_end in my_chunks:
        out_queue = ctx.Queue()
        p = ctx.Process(target=run_one_piece,
                         args=(api_key, chunk_start, chunk_end, out_queue),
                         daemon=True)
        p.start()
        try:
            result = out_queue.get(timeout=CALL_TIMEOUT_S)
        except queue.Empty:
            result = {"status": "timeout", "elapsed": CALL_TIMEOUT_S, "partial": {}}
        p.join(timeout=5)
        if p.is_alive():
            p.terminate()

        ok = result["status"] == "ok"
        with lock:
            if ok:
                for (pos, ref, alt), row in result["partial"].items():
                    for r in rows_by_position.get(pos, []):
                        if r["ref"] == ref and r["alt"] == alt:
                            r.update(row)
                write_csv_fn()
            counters["ok" if ok else "failed"] += 1
            done = counters["ok"] + counters["failed"]

        status_msg = f"ok  {result['elapsed']:.1f}s" if ok else \
            "FAILED -- a rerun will likely fix it"
        print(f"  [key {worker_id}] {CHROM}:{chunk_start}-{chunk_end}  "
              f"{status_msg}  ({done}/{counters['total']})", flush=True)

def _write_csv(rows_list, score_cols):
    tmp = SCORES_CSV + ".tmp"
    with open(tmp, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["position", "ref", "alt"] + score_cols)
        w.writeheader()
        w.writerows(rows_list)
    os.replace(tmp, SCORES_CSV)

def main() -> None:
    scorers = _build_scorers()
    score_cols = [c for c, _, _ in scorers]

    if not Path(SCORES_CSV).exists():
        print(f"Fetching reference sequence ({CHROM}:{ISM_START+1}-{ISM_END})...")
        ref_seq = fetch_reference_sequence(CHROM.replace("chr", ""), ISM_START + 1, ISM_END)
        rows_list = build_all_variant_rows(ref_seq, ISM_START + 1, score_cols)
        _write_csv(rows_list, score_cols)
        print(f"Pre-initialised {SCORES_CSV} with {len(rows_list):,} rows "
              f"({len(ref_seq):,} positions x 3 alts).")
    else:
        with open(SCORES_CSV, newline="") as f:
            reader = csv.DictReader(f)
            missing_cols = [c for c in score_cols if c not in reader.fieldnames]
            if missing_cols:
                raise SystemExit(
                    f"{SCORES_CSV} doesn't have column(s) {missing_cols} -- its "
                    f"scorer config doesn't match the current _build_scorers(). "
                    f"Delete the file or change the region to start fresh."
                )
            rows_list = [{**r, "position": int(r["position"])} for r in reader]
        print(f"Loaded existing {SCORES_CSV} ({len(rows_list):,} rows).")

    rows_by_position: dict = {}
    for r in rows_list:
        rows_by_position.setdefault(r["position"], []).append(r)

    def chunk_is_done(cs, ce):
        return not any(r[c] == ""
                        for pos in range(cs + 1, ce + 1)
                        for r in rows_by_position.get(pos, [])
                        for c in score_cols)

    chunks = [(p, min(p + CHUNK_BP, ISM_END)) for p in range(ISM_START, ISM_END, CHUNK_BP)]
    todo = [c for c in chunks if not chunk_is_done(*c)]
    print(f"{len(chunks)} chunks total | {len(chunks) - len(todo)} already done | "
          f"{len(todo)} to run\n")
    if not todo:
        print("All chunks complete.")
        return

    lock = threading.Lock()
    write_csv_fn = lambda: _write_csv(rows_list, score_cols)
    key_chunks = {i: todo[i::len(API_KEYS)] for i in range(len(API_KEYS))}
    counters = {"ok": 0, "failed": 0, "total": len(todo)}

    print(f"Launching {len(API_KEYS)} key-dispatchers over {len(todo)} chunks "
          f"({CHUNK_BP} bp each)...\n")
    run_start = time.time()

    threads = [
        threading.Thread(
            target=key_dispatcher,
            args=(i + 1, key, key_chunks[i], rows_by_position, lock, write_csv_fn, counters),
            daemon=True,
        )
        for i, key in enumerate(API_KEYS)
    ]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    elapsed_min = (time.time() - run_start) / 60
    print()
    print("=" * 60)
    print(f"Finished in {elapsed_min:.1f} min | {counters['ok']} chunks ok | "
          f"{counters['failed']} chunks left blank")
    print("Re-run the script to retry the blank cells." if counters["failed"]
          else "All chunks complete.")




In [11]:
model = dna_client.create(API_KEYS[0])
ism_interval = genome.Interval(CHROM, 156_035_791, 156_035_794)
LMNA_INTERVAL = genome.Interval('chr1', 156_082_572, 156_140_081)
seq_interval = LMNA_INTERVAL.resize(131072)
print(seq_interval)
scorers = _build_scorers()


result = model.score_ism_variants(
    interval=seq_interval,
    ism_interval=ism_interval,
    variant_scorers=[s for _, _, s in scorers],
    organism=dna_client.Organism.HOMO_SAPIENS,
    progress_bar=False,
    max_workers=MAX_WORKERS,
    merge_stranded_gene_tracks=False,
)


chr1:156045791-156176863:.


_MultiThreadedRendezvous: <_MultiThreadedRendezvous of RPC that terminated with:
	status = StatusCode.INVALID_ARGUMENT
	details = "ISM interval chr1:156035791-156035794:. not fully contained in interval=Interval(chromosome='chr1', start=156045791, end=156176863, strand='.', name='')."
	debug_error_string = "INVALID_ARGUMENT:ISM interval chr1:156035791-156035794:. not fully contained in interval=Interval(chromosome='chr1', start=156045791, end=156176863, strand='.', name='')."
>

In [15]:
len(result)

9

In [20]:
_build_scorers()

[('ATAC, RECOMMENDED',
  'ATAC',
  CenterMaskScorer(requested_output=ATAC, width=501, aggregation_type=DIFF_LOG2_SUM)),
 ('CONTACT_MAPS, RECOMMENDED', 'CONTACT_MAPS', ContactMapScorer()),
 ('DNASE, RECOMMENDED',
  'DNASE',
  CenterMaskScorer(requested_output=DNASE, width=501, aggregation_type=DIFF_LOG2_SUM)),
 ('CHIP_TF, RECOMMENDED',
  'CHIP_TF',
  CenterMaskScorer(requested_output=CHIP_TF, width=501, aggregation_type=DIFF_LOG2_SUM)),
 ('CHIP_HISTONE, RECOMMENDED',
  'CHIP_HISTONE',
  CenterMaskScorer(requested_output=CHIP_HISTONE, width=2001, aggregation_type=DIFF_LOG2_SUM)),
 ('CAGE, RECOMMENDED',
  'CAGE',
  CenterMaskScorer(requested_output=CAGE, width=501, aggregation_type=DIFF_LOG2_SUM)),
 ('PROCAP, RECOMMENDED',
  'PROCAP',
  CenterMaskScorer(requested_output=PROCAP, width=501, aggregation_type=DIFF_LOG2_SUM)),
 ('RNA_SEQ, RECOMMENDED',
  'RNA_SEQ',
  GeneMaskLFCScorer(requested_output=RNA_SEQ)),
 ('RNA_SEQ_ACTIVE, RECOMMENDED',
  'RNA_SEQ_ACTIVE',
  GeneMaskActiveScorer(reques

In [37]:
result[0][10].layers

Layers with keys: quantiles

In [16]:
def extract_lv(adata):
  values = adata.X[:, adata.var['ontology_curie'] == 'UBERON:0002084']
  assert values.size == 1
  return values.flatten()[0]


ism_result = ism.ism_matrix(
    [extract_lv(x[0]) for x in result],
    variants=[v[0].uns['variant'] for v in result],
)



In [17]:
ism_result

array([[ 0.        ,  0.        , -0.01614809, -0.        ],
       [-0.01526133,  0.        , -0.        ,  0.        ],
       [ 0.        ,  0.02783108,  0.        , -0.        ]],
      dtype=float32)